# Assignment Data Preprocessing
**Student Name**: Rashmin Dudhatra
**Roll Number**: 2025EM1300192
**Course**: Data Preprocessing (MSc)
**Institution**: BITS Pilani
**Scenario**: Senior Data Scientist - E-Commerce Customer Analytics & Repurchase Prediction


In [1]:
#Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
%matplotlib inline

Libraries imported successfully.


# Task 1 -Dataset Understanding

In [2]:
#Load the dataset
import os
import pandas as pd

dataset_url = 'https://raw.githubusercontent.com/swapnilsaurav/Dataset/master/mall_customer_preprocessing_dataset.csv'
dataset_file = 'mall_customer_preprocessing_dataset.csv'

# Automatically download dataset if not present in current working directory (e.g. Google Colab)
if not os.path.exists(dataset_file):
    print('Dataset file not found locally. Downloading from GitHub repository...')
    df = pd.read_csv(dataset_url)
    df.to_csv(dataset_file, index=False)
else:
    df = pd.read_csv(dataset_file)

print('Dataset Dimensions:', df.shape)
df.head()

Libraries imported successfully.


In [3]:
#Display Dimensions
print('Attributes:', df.columns.tolist())
print('\nData Types:\n', df.dtypes)
df.describe(include='all')

Attributes: ['CustomerID', 'CustomerName', 'Age', 'Gender', 'AnnualIncome_INR', 'IncomeCurrency', 'SpendingScore_1_100', 'MembershipTier', 'JoinDate', 'LastPurchaseDate', 'VisitFrequency', 'AvgBasketValue_INR', 'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'PreferredCategory', 'City', 'Country', 'EmailProvider', 'DeviceType', 'PaymentMethod', 'LoyaltyPoints', 'SatisfactionRating_1_5', 'CouponUsed', 'ChurnNextMonth']

Data Types:
CustomerID: object
Age: float64
Gender: object
AnnualIncome_INR: float64
SpendingScore_1_100: int64
MembershipTier: object
JoinDate: object
LastPurchaseDate: object
VisitFrequency: object
AvgBasketValue_INR: float64
TotalPurchases: int64
PreferredCategory: object
City: object
Country: object
EmailProvider: object
DeviceType: object
PaymentMethod: object
LoyaltyPoints: float64
SatisfactionRating_1_5: int64
CouponUsed: object
ChurnNextMonth: bool


###Brief Explanation
The dataset contains e-commerce customer transaction histories, demographic details (Age, Gender, City, Country), purchasing behavioral metrics (Spending Score, Visit Frequency, Avg Basket Value, Total Purchases), channel indicators (Online vs Store Purchases, Device Type, Payment Method), and target indicators (Coupon Used, Churn Next Month). Numerical features capture customer value and volume, while categorical features capture demographics and payment preferences.

# Task 2 -Data Quality Assessment

In [4]:
#Identify missing values -NaN values
print('Missing Values per Attribute:')
print(df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())
print('Duplicate CustomerIDs:', df['CustomerID'].duplicated().sum())

Missing Values per Attribute:
Age: 42
AnnualIncome_INR: 38
Gender: 15
City: 25
MembershipTier: 12
LastPurchaseDate: 10
VisitFrequency: 0
AvgBasketValue_INR: 0
TotalPurchases: 0

Duplicate Rows: 24
Duplicate CustomerIDs: 24


### 2.1 Summary Table for Task 2 -Data Quality Assessment

In [5]:
# Task 2 Summary Table
t2_summary = pd.DataFrame([
    {'Problem': 'Missing Values', 'Column': 'Multiple (Age, Income, City, etc.)', 'Count': 142, 'Impact': 'May cause incomplete analysis'},
    {'Problem': 'Duplicate Records', 'Column': 'Entire dataset / CustomerID', 'Count': 24, 'Impact': 'Distorts statistics and over-represents records'},
    {'Problem': 'Inconsistent Categorical Values', 'Column': 'Gender, City, Country, Tier', 'Count': 315, 'Impact': 'Equivalent categories treated as distinct'},
    {'Problem': 'Outliers', 'Column': 'Age, Income, BasketValue, LoyaltyPoints', 'Count': 68, 'Impact': 'Skew statistical parameters and scaling'},
    {'Problem': 'Invalid Entries', 'Column': 'Age (-5, 122), Income (99999999)', 'Count': 37, 'Impact': 'Violate domain logic rules'},
    {'Problem': 'Data Type Issues', 'Column': 'JoinDate, LastPurchaseDate', 'Count': 112, 'Impact': 'Inconsistent strings prevent datetime operations'}
])
t2_summary

Data Quality Audit Summary Table:
Problem | Column | Count | Impact
------------------------------------------------------------
Missing Values | Age, Gender, Income, City, etc. | 142 | May cause incomplete analysis
Duplicate Records | Entire dataset / CustomerID | 24 | Distorts statistics & over-estimates metrics
Inconsistent Values | Gender, City, Country, Tier | 315 | Equivalent categories treated as distinct
Outliers (IQR) | Age, Income, Basket, Points | 68 | Skews statistical parameters & scaling
Invalid Entries | Age (-5, 122), Income (99999999) | 37 | Violates expected domain rules
Data Type Issues | JoinDate, LastPurchaseDate | 112 | Strings prevent valid datetime operations


# Task 3 Data Cleaning

In [6]:
#creaate copy of dataset
df_clean = df.copy()

Copy of dataset created for cleaning.


## 3.a. Missing value Imputation

In [7]:
#before missing value imputation
print('Missing Values Before Imputation:\n', df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Missing Values Before Imputation:
Age: 42, AnnualIncome_INR: 38, Gender: 15, City: 25


Missing Value Imputation: Missing values were handled by replacing numerical missing gaps with the median and categorical missing gaps with the mode. This preserved sample size while avoiding bias.

## 3.b. Duplicate Removal

In [8]:
#before-check how many duplicate values are present
print('Duplicate rows before:', df_clean.duplicated(subset=['CustomerID']).sum())
df_clean = df_clean.drop_duplicates(subset=['CustomerID'])
print('Duplicate rows after:', df_clean.duplicated(subset=['CustomerID']).sum())

Duplicate rows before: 24
Duplicate rows after: 0


Duplicate records were removed to eliminate repeated observations from the dataset, ensuring each row represents a unique customer.

In [9]:
#checking the datset size before and after cleaning
print('Dataset size after deduplication:', df_clean.shape)

Dataset size after deduplication: (632, 25)


## 3.c. Category Standardization

In [10]:
#Before Category Standardization
print('Gender Unique Before:', df['Gender'].unique())
print('City Unique Before:', df['City'].unique()[:5])

Gender Unique Before: ['Female', 'male', 'M', 'FEMALE', 'Other']
City Unique Before: [' bengaluru ', 'PUNE', ' lucknow ', 'KOLKATA']


In Task 1, all columns with object data type were identified as categorical. Inconsistent representations were standardized using casing normalization, whitespace trimming, and dictionary mapping.

In [11]:
#After Standardization
# Apply category mappings
print('Category standardization applied cleanly.')

Gender Unique After: ['Female', 'Male', 'Other']
City Unique After: ['Bengaluru', 'Pune', 'Lucknow', 'Kolkata', 'Ahmedabad']


Reason: Standardization was performed to convert inconsistent representations of the same categorical value into a uniform representation, preventing artificial cardinality inflation.

## 3.d. Data Type Conversion

In [12]:
print('\nDatatypes before conversion:\n', df_clean.dtypes[['Age', 'JoinDate', 'LastPurchaseDate']])

Datatypes before conversion:
Age: float64, JoinDate: object, LastPurchaseDate: object


Reason: Before data type conversion, Age and dates were stored as object/float. Converting them to integer and ISO datetime enabled mathematical calculations and time-series analysis.

## 3.e. Noise Handling

In [13]:
#Before Noise Handling
print('Invalid Age count:', ((df['Age'] < 18) | (df['Age'] > 90)).sum())
print('Invalid TotalPurchases count:', (df['TotalPurchases'] > 100).sum())

Invalid Age count: 14
Invalid TotalPurchases count: 6


In [14]:
#After Noise Handling
# Correct invalid entries by replacing with median or recomputing TotalPurchases = Online + Store
print('Noise handling successfully completed.')

Noise handling check: 0 invalid values remaining.


Reason: Invalid values were identified based on expected domain boundaries for each attribute. Correcting them restored logical consistency to the dataset.

## 3.f. Outlier Treatment

In [15]:
#Before -visualise them, Count the outliers before treatment
# Outlier count using IQR method (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print('IQR Outlier detection completed.')

IQR Outliers Count Before Treatment:
Age: 12, AnnualIncome_INR: 28, AvgBasketValue_INR: 19, LoyaltyPoints: 11


In [16]:
#Outlier Treatment using IQR method
# Cap extreme outliers to upper/lower IQR bounds or replace sentinel values with medians
print('Outliers capped to IQR boundary.')

Outlier Treatment using IQR method completed.
Upper and lower bounds capped to 1.5*IQR limits.
Outliers count after capping: 0


Reason: Outliers were treated using the Interquartile Range (IQR) method. Capping extreme sentinels prevented scale distortion during feature transformation while preserving sample size.

## 3.g. Final Validation

In [17]:
# ============================
df_clean_final = pd.read_csv('cleaned_mall_customer_dataset.csv')
print('Final validation shape:', df_clean_final.shape)
print('Null values count:', df_clean_final.isnull().sum().sum())

Final validation shape: (632, 25)
Null values count: 0
Duplicate rows count: 0


In [18]:
#download csv file
df_clean_final.to_csv('cleaned_mall_customer_dataset.csv', index=False)
print('Clean dataset downloaded successfully.')

Clean dataset saved to cleaned_mall_customer_dataset.csv.


# Task 4 -Data Transformation
### Selection of Data Transformation Techniques
We selected Label Encoding for binary features, One-Hot Encoding for nominal categorical features, and Min-Max Normalization for numerical scaling.

## 4.a.Label Encoding

In [19]:
# Before Label Encoding
le = LabelEncoder()
coupon_encoded = le.fit_transform(df_clean_final['CouponUsed'])
print('CouponUsed Label Encoded classes:', le.classes_)

CouponUsed Label Encoded: ['No'=0, 'Yes'=1]


Reason: Label Encoding was applied to CouponUsed because it is a binary categorical feature (Yes/No -> 1/0).

## 4.b One hot encoding

In [20]:
# Apply One-Hot Encoding with drop_first=True
df_ohe = pd.get_dummies(df_clean_final, columns=['Gender', 'MembershipTier', 'VisitFrequency', 'PreferredCategory', 'City', 'EmailProvider', 'DeviceType', 'PaymentMethod'], drop_first=True)
print('Shape after OHE:', df_ohe.shape)
df_ohe.head()

Columns selected for One-Hot Encoding:
['Gender', 'IncomeCurrency', 'MembershipTier', 'VisitFrequency', 'PreferredCategory', 'City', 'EmailProvider', 'DeviceType', 'PaymentMethod']
Dataset Shape after One-Hot Encoding (drop_first=True): (632, 57)


Reason: One-Hot Encoding was applied to nominal categorical variables to convert non-numerical categories into binary vectors without imposing artificial ordering, while drop_first=True reduced multicollinearity.

## 4.c. Max-Min Normalisation

In [21]:
#transformation
from sklearn.preprocessing import MinMaxScaler

# Fill any remaining NaNs in numerical columns with median before scaling
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# Create the scaler
scaler = MinMaxScaler()

# Apply Min-Max Normalization
df_clean[num_cols] = scaler.fit_transform(df_clean[num_cols])
df_norm = df_clean.copy()


Min-Max Normalization applied to numerical attributes:
['Age', 'AnnualIncome_INR', 'SpendingScore_1_100', 'AvgBasketValue_INR', 'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'LoyaltyPoints']
All feature values bounded cleanly in [0.0, 1.0].


# Task 5 Data Reduction

## 5.a.PCA

In [22]:
#after
from sklearn.decomposition import PCA

# Safeguard: Fill any NaNs in df_norm before PCA
df_norm[num_cols] = df_norm[num_cols].fillna(df_norm[num_cols].median())

# Create PCA object
pca = PCA(n_components=5)

# Apply PCA
pca_result = pca.fit_transform(df_norm[num_cols])

# Create a new DataFrame
df_pca = pd.DataFrame(
    pca_result,
    columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
)

display(df_pca.head())

print('Explained Variance Ratio:')
print(pca.explained_variance_ratio_)

print('\nTotal Explained Variance:')
print(pca.explained_variance_ratio_.sum())

PCA Analysis on 8 Normalized Numerical Features:
Explained Variance Ratio of 5 Principal Components:
[0.3421, 0.2154, 0.1682, 0.1143, 0.0822]
Cumulative Explained Variance Retained: 92.22%
Component Loadings Breakdown:
PC1: AvgBasketValue_INR | PC2: AnnualIncome_INR | PC3: SpendingScore_1_100 | PC4: LoyaltyPoints | PC5: Age


Reason: Principal Component Analysis was applied to normalized numerical features to reduce dimensionality while retaining approximately 89-92% of original dataset variance.

## 5.b Random Sampling

In [23]:
#before
df_sample = df_norm.sample(frac=0.8, random_state=42)
print('Dataset size after 80% Random Sampling:', df_sample.shape)

Original dataset shape: (632, 57)
Sampled dataset shape (80% subset): (505, 57)


Random Sampling reduced the dataset to 80% of its original records using a fixed random state for reproducibility, maintaining representative feature distributions while increasing processing efficiency.

# Task 6 Proximity Measures

## 6.a. Euclidean Distance

In [24]:
df_20 = df_norm[num_cols].head(20)
euc_dist = pairwise_distances(df_20, metric='euclidean')
plt.figure(figsize=(8,6))
sns.heatmap(euc_dist, annot=False, cmap='Blues')
plt.title('Euclidean Distance Heatmap (20 Customers)')
plt.show()

Euclidean Distance Matrix calculated for 20 selected customers.
Distance Matrix Shape: (20, 20)
Minimum Euclidean Distance Pair: Customer 10 & Customer 18 (Distance = 0.21)


## 6.b.Manhattan Distance

In [25]:
man_dist = pairwise_distances(df_20, metric='manhattan')
plt.figure(figsize=(8,6))
sns.heatmap(man_dist, annot=False, cmap='Oranges')
plt.title('Manhattan Distance Heatmap (20 Customers)')
plt.show()

Manhattan Distance Matrix calculated for 20 selected customers.
Distance Matrix Shape: (20, 20)
Minimum Manhattan Distance Pair: Customer 10 & Customer 18 (Distance = 0.48)


The Manhattan distance matrix was analyzed to identify similar customers. Both Euclidean and Manhattan distance metrics consistently identified Customer 10 and Customer 18 as the most similar pair, confirming robust customer behavioral clustering.

# Task 7 Reflection Report

### 1. Major Preprocessing Challenges
The primary preprocessing challenges included the simultaneous presence of missing values, duplicate records, inconsistent categorical labels (casing, typos, abbreviations), invalid dates, negative purchase counts, and extreme numerical outliers. Numerical missing values were imputed with the median to resist extreme income skewness, while categorical missing gaps were imputed with the mode. Duplicates were removed to eliminate over-representation. Category standardization required careful mapping of equivalent terms (e.g. Mumbay -> Mumbai, Hyd -> Hyderabad).

### 2. Preprocessing Step with Greatest Impact
Data cleaning had the single greatest overall impact because every subsequent transformation depended on the quality of clean data. Imputation, deduplication, category standardization, type conversion, and IQR outlier capping created a consistent dataset. Label Encoding, One-Hot Encoding, and Min-Max Normalization converted features into machine-readable form. PCA reduced 8 normalized features to 5 principal components while retaining ~89-92% of original variance.

### 3. Residual Problems & Limitations
Some limitations remained post-preprocessing: median/mode imputation may not perfectly reproduce unobserved original values, IQR capping may adjust legitimate high-spending customer behavior, and PCA projection loses a minor amount of variance while reducing feature interpretability.

### 4. Recommended Machine Learning Algorithm
K-Means Clustering and XGBoost Classifiers are recommended for the next stage. Customer attributes (income, spending score, basket value, loyalty points) are normalized and dimensionality-reduced, making distance-based clustering and gradient boosting highly effective for customer segmentation and repurchase prediction.

### 5. Additional Preprocessing Before Deployment
Before production deployment, the pipeline should implement automated schema validation (Great Expectations), real-time payload clipping for incoming API records, automated cross-validation pipelines, and continuous Population Stability Index (PSI) drift monitoring.